# apertus-eval-prep — ranking stability on Colab

Runtime → Change runtime type → **T4 GPU**.

Do **not** Run all. One session = setup cells + **one** sweep cell + the save at the bottom of that cell.

Free Colab dies after 1–2 hours **or** when GPU quota is exhausted. Each item is written to Drive as `{run_id}.partial.jsonl`. If the runtime dies at `[520/800]`, the next session resumes at 521 — **after** `git pull` so that checkpoint code is installed.

Do not treat notebook stdout as a result. Only `results/runs/*.json` plus a registry row is a finished cell.

Next session: setup (Drive restore), then the **same** sweep cell if a `.partial.jsonl` exists, else the next unfinished cell. Finished `config_hash` rows skip.

Use `results/registry_paper.jsonl` (not the n=4 smoke `registry.jsonl`).

In [ ]:
import os
if os.path.exists("pyproject.toml") and os.path.exists("data/eval_set.jsonl"):
    print("Already in repo root")
    !git pull --ff-only
elif os.path.exists("apertus-eval-prep/pyproject.toml"):
    %cd apertus-eval-prep
    !git pull --ff-only
else:
    !git clone https://github.com/Shivani767/apertus-eval-prep.git
    %cd apertus-eval-prep
!pip -q install -e ".[gpu,viz]"
!git log -1 --oneline

In [ ]:
import torch
from pathlib import Path
assert torch.cuda.is_available(), "Set runtime to GPU (T4) and rerun."
print(torch.cuda.get_device_name(0))
if not Path("data/official/eval_set.jsonl").exists():
    !pip -q install -e ".[snapshot]"
    !python scripts/snapshot_benchmarks.py
else:
    print("official slices already on disk")

## Persist to Google Drive

Authorize Drive when prompted. Files live in `MyDrive/apertus-eval-prep-paper/` so a runtime reset does not wipe finished cells.

Also download the zip to your Mac as a second copy.

In [ ]:
import os, sys
from pathlib import Path
from google.colab import drive, files

def ensure_repo():
    for p in (Path.cwd(), Path("/content/apertus-eval-prep"), Path("apertus-eval-prep")):
        if (p / "pyproject.toml").exists() and (p / "src" / "apertus_eval_prep").exists():
            os.chdir(p)
            src = str((p / "src").resolve())
            if src not in sys.path:
                sys.path.insert(0, src)
            print("cwd:", Path.cwd(), flush=True)
            return p
    raise FileNotFoundError("Repo not found. Run the clone/pip cell first.")

ensure_repo()
try:
    import apertus_eval_prep  # noqa: F401
except ModuleNotFoundError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[gpu,viz]"])

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/apertus-eval-prep-paper")
DRIVE.mkdir(parents=True, exist_ok=True)
(DRIVE / "runs").mkdir(exist_ok=True)
Path("results/runs").mkdir(parents=True, exist_ok=True)
os.environ["APERTUS_CHECKPOINT_DIR"] = str(DRIVE / "runs")

if (DRIVE / "registry_paper.jsonl").exists():
    !cp -a {DRIVE}/registry_paper.jsonl results/registry_paper.jsonl
    print("restored registry from Drive")
if any((DRIVE / "runs").iterdir()):
    !cp -a {DRIVE}/runs/. results/runs/
    print("restored runs from Drive")
n_partial = len(list(Path("results/runs").glob("*.partial.jsonl")))
print(f"partial checkpoints on disk: {n_partial}", flush=True)

def save_paper():
    import subprocess
    DRIVE.mkdir(parents=True, exist_ok=True)
    (DRIVE / "runs").mkdir(exist_ok=True)
    if Path("results/registry_paper.jsonl").exists():
        subprocess.check_call(["cp", "-a", "results/registry_paper.jsonl", str(DRIVE / "registry_paper.jsonl")])
    if Path("results/runs").exists():
        subprocess.check_call(["bash", "-lc", f"cp -a results/runs/. {DRIVE}/runs/"])
    n = len(list(Path("results/runs").glob("*.json")))
    print(f"saved to Drive ({n} run JSON files):", DRIVE, flush=True)
    subprocess.check_call(["zip", "-r", "/tmp/paper_matrix_partial.zip", "results/runs", "results/registry_paper.jsonl"])
    files.download("/tmp/paper_matrix_partial.zip")

def sweep(*extra):
    """Run in-process so Colab shows [1/800] live. subprocess.check_call hid all logs."""
    ensure_repo()
    only_model = only_factor = None
    args = list(extra)
    i = 0
    while i < len(args):
        if args[i] == "--only-model" and i + 1 < len(args):
            only_model = args[i + 1]
            i += 2
        elif args[i] == "--only-factor" and i + 1 < len(args):
            only_factor = args[i + 1]
            i += 2
        else:
            raise ValueError(f"unknown sweep arg {args[i]!r}")
    from apertus_eval_prep.sweep import execute_sweep
    print(f"sweep in-process model={only_model} factor={only_factor}", flush=True)
    planned = execute_sweep(
        study_path=Path("configs/experiments/stability.yaml"),
        repo_root=Path(".").resolve(),
        out_dir=Path("results/runs"),
        registry_path=Path("results/registry_paper.jsonl"),
        profile="t4",
        only_model=only_model,
        only_factor=only_factor,
    )
    n_skip = sum(1 for p in planned if p["skipped"])
    print({"n_cells": len(planned), "n_skip": n_skip}, flush=True)
    save_paper()

!python -m apertus_eval_prep sweep --config configs/experiments/stability.yaml --profile t4 --dry-run --out-dir results/runs --registry results/registry_paper.jsonl | head -n 50

## Session cells (run one per Colab window)

**Already on GitHub:** SmolLM2 control (318/800), Qwen2.5-3B control (515/800), Phi-3.5-mini control (536/800). Those three cells should `skip`.

| When | Cell |
|---|---|
| Done | SmolLM2 / Qwen 3B / Phi **control** |
| **This GPU** | SmolLM2 `prompt_id` |
| **Other Google account** | Qwen 3B `prompt_id` (cell below) |
| Later | Phi `prompt_id`, then other factors |
| Later | remaining factors / 7B int4 |

Wait until `[800/800]` and the Drive zip download finish before closing the tab. If Colab says GPU time is up, stop. Next GPU window: git pull, Drive cell, **the same unfinished cell** (resumes from `.partial.jsonl`).

In [ ]:
# DONE on GitHub — skip. Only run if dry-run says run not skip.
sweep("--only-model", "HuggingFaceTB/SmolLM2-1.7B-Instruct", "--only-factor", "control")

In [ ]:
# DONE on GitHub — Qwen 3B control (515/800). Skip unless dry-run says run.
sweep("--only-model", "Qwen/Qwen2.5-3B-Instruct", "--only-factor", "control")

In [ ]:
# DONE on GitHub — Phi-3.5 control (536/800). Skip unless dry-run says run.
sweep("--only-model", "microsoft/Phi-3.5-mini-instruct", "--only-factor", "control")

In [ ]:
# DONE on GitHub — SmolLM2 prompt_id (concise 186/800, 5shot 274/800). Skip unless dry-run says run.
sweep("--only-model", "HuggingFaceTB/SmolLM2-1.7B-Instruct", "--only-factor", "prompt_id")

In [ ]:
# Other account / next GPU — Qwen 3B prompt_id only (concise + 5shot). Two 800-item runs.
# Do not run this on the same runtime as SmolLM2 prompt_id.
sweep("--only-model", "Qwen/Qwen2.5-3B-Instruct", "--only-factor", "prompt_id")

In [ ]:
# Later GPU — Phi-3.5 prompt_id only (concise + 5shot). Two 800-item runs.
sweep("--only-model", "microsoft/Phi-3.5-mini-instruct", "--only-factor", "prompt_id")

In [ ]:
# Later — remaining SmolLM2 factors (control is skipped)
sweep("--only-model", "HuggingFaceTB/SmolLM2-1.7B-Instruct")

In [ ]:
# Later — remaining Qwen 3B factors
sweep("--only-model", "Qwen/Qwen2.5-3B-Instruct")

In [ ]:
# Later — remaining Phi-3.5 factors
sweep("--only-model", "microsoft/Phi-3.5-mini-instruct")

In [ ]:
# T4 only keeps 7B int4 (fp16 / int8 / vLLM are skipped)
sweep("--only-model", "Qwen/Qwen2.5-7B-Instruct")

## Report (only after several cells exist)

cwd must be `/content/apertus-eval-prep`. Do not run this instead of a sweep.

In [ ]:
from google.colab import files
from pathlib import Path

assert Path("results/registry_paper.jsonl").exists(), "No paper registry in this runtime. Restore from Drive first."
!python -m apertus_eval_prep report --registry results/registry_paper.jsonl --out reports/stability_paper
!python -m apertus_eval_prep paper-tables --registry results/registry_paper.jsonl --out paper/_generated_tables.md
!zip -r paper_matrix_artifacts.zip results/runs results/registry_paper.jsonl reports/stability_paper paper/_generated_tables.md
print("zip bytes", Path("paper_matrix_artifacts.zip").stat().st_size)
files.download("paper_matrix_artifacts.zip")

Unpack the zip (or Drive folder) into the Mac clone. Commit `results/registry_paper.jsonl` and new `results/runs/*.json`. Do not edit numbers.